# AdS4 Dual-Solution Discovery (v15)

Constraint swap test:
- Same EE ⊕ WL
- Two GEO constraints
→ check if both produce stable but different solutions


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
r = torch.linspace(1.05, 3.5, 240, device=device).view(-1, 1)

def f_true(r): return r**2 + 1 - 1/r
def g_true(r): return 1/(f_true(r) + 1e-6)

def ee(f,g): return torch.log(1+0.6*f)
def wl(f,g): return torch.sqrt(f+1.8*g+1e-6)
def geo(f,g): return torch.sqrt(f+0.7*g+1e-6)

ee_obs = ee(f_true(r), g_true(r)).detach()
wl_obs = wl(f_true(r), g_true(r)).detach()
geo_base = geo(f_true(r), g_true(r)).detach()


In [ ]:
def geo_A(r): return geo_base
def geo_B(r):
    return geo_base * (1+0.25*(r-r.min())/(r.max()-r.min())) * (1+0.1*torch.sin(2*r))


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh(),nn.Linear(64,1))
    def forward(self,x): return self.net(x)

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.f=MLP(); self.g=MLP()
    def forward(self,r):
        return F.softplus(self.f(r)), F.softplus(self.g(r))


In [ ]:
def train(geo_fn):
    m=Model().to(device)
    opt=torch.optim.Adam(m.parameters(),lr=2e-3)
    target=geo_fn(r)
    loss_hist=[]
    for _ in range(1200):
        opt.zero_grad()
        f,g=m(r)
        loss=((ee(f,g)-ee_obs)**2).mean()+((wl(f,g)-wl_obs)**2).mean()+((geo(f,g)-target)**2).mean()+0.15*((f*g-1)**2).mean()
        loss.backward(); opt.step(); loss_hist.append(loss.item())
    return m, loss_hist


In [ ]:
mA, lA = train(geo_A)
mB, lB = train(geo_B)

with torch.no_grad():
    fA,gA=mA(r); fB,gB=mB(r)


In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(r.cpu(), f_true(r).cpu(),'k',label='true')
plt.plot(r.cpu(), fA.cpu(),label='A')
plt.plot(r.cpu(), fB.cpu(),label='B')
plt.title('f(r)'); plt.legend()

plt.subplot(1,2,2)
plt.plot(r.cpu(), g_true(r).cpu(),'k',label='true')
plt.plot(r.cpu(), gA.cpu(),label='A')
plt.plot(r.cpu(), gB.cpu(),label='B')
plt.title('g(r)'); plt.legend()

plt.tight_layout()
plt.savefig('ads4_phase_lock_v15_dual.png',dpi=180)
plt.show()
print('Saved: ads4_phase_lock_v15_dual.png')